# Exercise 04 : A Multi-Agent Visit Planner

- A fixed workflow runs two specialists in parallel ; `Send` gives each one its own task
- Each specialist is its own agent, with its own tool and its own message list
- A checker written in code checks specified properties without a model call
- The planner is compared with one agent on calls and tokens

You will plan one visit to site S04. Devices come from the database. A parts
agent and a manuals agent run in parallel. A checker reads the plan, then one
agent does the same task. The Aurelia records are made up.

Worked examples to revisit : module 05, demo03b · router-parallel, demo06b · workflows-parallel,
demo07 · maker-checker, demo09 · what-each-pattern-costs.

Run the next cell first. Its code is folded : click the cell and press Shift+Enter. It loads the course key and builds `llm`, the course model.

In [ ]:
#@title Course setup : run this cell first
# On the classroom laptop the key comes from .env in the repository root.
# In Colab, add it under the key icon in the left sidebar as COURSE_API_KEY.
import os
import sys
import subprocess
import warnings
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
COURSE_FILES = "https://raw.githubusercontent.com/FeikoWielsma/Multi-Agents-LangGraph-Student/main/"
ROOT = None

if IN_COLAB:
    # Pinned to match uv.lock ; generated by build/stamp_setup.py
    PINS = ["langchain[mcp]==1.4.6", "langchain-core==1.6.12", "langchain-openai==1.7.1", "langgraph==1.2.17", "langgraph-checkpoint-sqlite==3.1.2", "deepagents[quickjs]==0.7.30", "langchain-quickjs==0.3.9", "langsmith==0.14.12", "openai==3.29.0", "python-dotenv==1.2.4", "langchain-huggingface==1.2.3", "sentence-transformers==6.1.0", "langchain-text-splitters==1.1.3", "rank-bm25==0.2.2", "agentevals==0.0.9", "httpx==0.28.1", "pytest==9.1.1", "pandas==3.0.6"]
    if not os.getenv("COURSE_SKIP_INSTALL"):
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINS], check=True)
    try:
        from google.colab import userdata
        os.environ.setdefault("COURSE_API_KEY", userdata.get("COURSE_API_KEY"))
    except Exception:
        pass
else:
    from dotenv import load_dotenv
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / "pyproject.toml").exists():
            ROOT = folder
            load_dotenv(folder / ".env")
            break

PROVIDER = os.getenv("COURSE_PROVIDER", "course")      # "course" or "ollama"
if PROVIDER == "ollama":
    MODEL = os.getenv("COURSE_MODEL", "qwen3.5:4b")
    BASE_URL = "http://localhost:11434/v1"
    API_KEY = "ollama"
    THINKING_OFF = {"reasoning_effort": "none"}
else:
    MODEL = os.getenv("COURSE_MODEL", "openai/gpt-6-luna")  # xiaomi/mimo-v2.6-pro is the step up
    BASE_URL = os.getenv("COURSE_BASE_URL", "https://openrouter.ai/api/v1")
    API_KEY = os.getenv("COURSE_API_KEY")
    # OpenRouter has no Luna endpoint that honours temperature or parallel_tool_calls ;
    # seed is honoured, and routing prefers OpenAI so repeated runs hit one provider
    THINKING_OFF = {"reasoning": {"effort": "none"}, "provider": {"order": ["openai"]}}
    if not API_KEY:
        raise RuntimeError("COURSE_API_KEY is missing : copy .env.example to .env and paste the key")

from langchain_core._api import LangChainBetaWarning
warnings.filterwarnings("ignore", category=LangChainBetaWarning)

from langchain_openai import ChatOpenAI
from langchain_core.rate_limiters import InMemoryRateLimiter

# One shared limiter paces every model call in this process ; COURSE_RPS sets the budget.
# Paid OpenRouter models have no platform request cap (measured 26 October 2026)
RATE_LIMITER = InMemoryRateLimiter(requests_per_second=float(os.getenv("COURSE_RPS", "2")),
                                   check_every_n_seconds=0.1, max_bucket_size=1)


def make_llm(**kw):
    """The course chat model. Keyword arguments pass straight to ChatOpenAI."""
    kw.setdefault("model", MODEL)
    if PROVIDER == "ollama":
        kw.setdefault("temperature", 0)
    else:
        kw.setdefault("seed", 7)
    kw.setdefault("extra_body", THINKING_OFF)
    kw.setdefault("timeout", 90)        # seconds per request ; a stuck call raises instead of hanging
    kw.setdefault("rate_limiter", RATE_LIMITER)
    return ChatOpenAI(base_url=BASE_URL, api_key=API_KEY, **kw)


def make_embeddings(**kw):
    """Local sentence embeddings, so retrieval costs no API calls."""
    from langchain_huggingface import HuggingFaceEmbeddings
    kw.setdefault("model_name", "sentence-transformers/all-MiniLM-L6-v2")
    return HuggingFaceEmbeddings(**kw)


def data(name):
    """Path to a course data file ; downloaded when there is no local copy."""
    if ROOT and (ROOT / "data" / name).exists():
        return str(ROOT / "data" / name)
    target = Path(name)
    if not target.exists():
        import urllib.request
        target.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(COURSE_FILES + "data/" + name, target)
    return str(target)


def data_folder(name):
    """Path to a course data folder ; downloaded file by file, using its index.txt, when there is no local copy."""
    if ROOT and (ROOT / "data" / name).is_dir():
        return str(ROOT / "data" / name)
    for file in Path(data(f"{name}/index.txt")).read_text(encoding="utf-8").split():
        data(f"{name}/{file}")
    return name


llm = make_llm()
print(f"model {MODEL} via {PROVIDER}")

## Step 1 : Tools

`run_sql` and `search_manuals` read the Aurelia database and manuals. Both specialists and the
single agent use them, so this step confirms they work. It has no blanks.

The docstring of `run_sql` lists every table and column, which the model reads when it writes a
query. The first cell also prepares Step 4 : `metered` is the model that counts calls and tokens, and `measured` runs one
design and prints one `metrics :` line.

The second cell lists the devices at S04 and tries a statement that is not a SELECT, and `check
: pass` confirms both results. A `NameError` on `data` means the setup cell has not run.

In [ ]:
import json
import operator
import re
import sqlite3
import threading
import time
from pathlib import Path
from typing import Annotated, TypedDict

from langchain.agents import create_agent
from langchain.messages import AIMessage, HumanMessage, SystemMessage
from langchain.tools import tool
from langchain_core.callbacks import BaseCallbackHandler
from langgraph.errors import GraphRecursionError
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.types import Send


class Counter(BaseCallbackHandler):
    def __init__(self):
        self.lock = threading.Lock()
        self.reset()

    def reset(self):
        self.calls = self.input = self.output = 0

    def on_chat_model_start(self, serialized, messages, **kwargs):
        with self.lock:
            self.calls += 1

    def on_llm_end(self, response, **kwargs):
        with self.lock:
            for generations in response.generations:
                for generation in generations:
                    message = getattr(generation, "message", None)
                    usage = getattr(message, "usage_metadata", None) or {}
                    self.input += usage.get("input_tokens", 0)
                    self.output += usage.get("output_tokens", 0)


counter = Counter()
metered = make_llm(callbacks=[counter], max_tokens=600, max_retries=0, timeout=120)
metrics = []


def measured(label, operation):
    counter.reset()
    start = time.perf_counter()
    try:
        return operation()
    finally:
        row = {"pattern": label, "calls": counter.calls, "input_tokens": counter.input,
               "output_tokens": counter.output, "seconds": round(time.perf_counter() - start, 1)}
        metrics.append(row)
        print("metrics :", json.dumps(row))


DB = Path(data("capstone/service.db")).resolve()
MANUALS = Path(data_folder("capstone/manuals"))
PAGES = {p.name: p.read_text(encoding="utf-8") for p in sorted(MANUALS.glob("*.md"))}


@tool
def run_sql(query: str) -> str:
    """Run a read-only SELECT. Tables: sites(site_id, name, city, contract),
    devices(serial, model, site_id, installed, firmware),
    parts(part_no, model, name, stock, price_eur, depot),
    work_orders(wo_id, serial, opened, status, error_code, summary, engineer_id)."""
    if not query.lstrip().lower().startswith("select"):
        return "Only SELECT statements are allowed."
    with sqlite3.connect(f"file:{DB.as_posix()}?mode=ro", uri=True) as db:
        return json.dumps(db.execute(query).fetchmany(15))


@tool
def search_manuals(query: str) -> str:
    """Find Aurelia manuals and bulletins. Returns the opening of the two best matches."""
    words = set(query.lower().split())
    ranked = sorted(PAGES, key=lambda name: -sum(PAGES[name].lower().count(w) for w in words))[:2]
    return "\n\n".join(f"[{name}]\n{PAGES[name][:800]}" for name in ranked)

In [ ]:
listed = run_sql.invoke({"query": "SELECT serial FROM devices WHERE site_id = 'S04'"})
blocked = run_sql.invoke({"query": "DELETE FROM devices"})
print("devices :", listed)
print("blocked :", blocked)
ok = "MR150-2024-0098" in listed and "UX20-2025-2311" in listed and blocked.startswith("Only SELECT")
print("check :", "pass" if ok else "fail")

## Step 2 : Checker

A checker written in code tests stated facts on every plan with no model call, and it proves
only what it tests, as the slide *Checkers Written in Code* shows. Like `check_note` in module
05 demo07b · checker-in-code, `check_plan` reads the facts from the database, tests the plan line
by line and returns a list of problems, empty when the plan passes.

The cell gives you the reading, the loops and the problem messages. `serials` lists the devices
at the site and `stock` is a dictionary from part number to units in stock. `PART` is a pattern
for part numbers, and `PART.findall(line)` gives every part number on one line ; error codes such
as `E-MR-202` do not match it. You write the three conditions that decide what is a problem :

- The first blank : the serial `s` does not appear in `plan`
- The second blank : `part`, such as `MR-CH-02`, is not a key of `stock`
- The third blank : `part` has stock 0 and `line` does not say "out of stock". Compare the line
  in lower case, because a model may write `Out of stock`

The cell only defines the function, so a blank left in shows as a `NameError` in the test cell
below it. That cell must pass before you go on. `the good plan should pass` means a condition is
too strict, and `the bad plan has three problems` means a condition misses a problem. When you
are stuck, open the collapsed Hint under the cell first, then the Solution.

In [ ]:
import re


PART = re.compile(r"\b[A-Z]{2}-[A-Z]{2}-\d{2}\b")


def check_plan(plan: str, site_id: str) -> list[str]:
    with sqlite3.connect(f"file:{DB.as_posix()}?mode=ro", uri=True) as db:
        serials = [s for (s,) in db.execute("SELECT serial FROM devices WHERE site_id = ?", (site_id,))]
        stock = dict(db.execute("SELECT part_no, stock FROM parts"))
    problems = [f"device {s} is missing" for s in serials if ____]
    for line in plan.splitlines():
        for part in PART.findall(line):
            if ____:
                problems.append(f"part {part} does not exist")
            elif ____:
                problems.append(f"part {part} has no stock and the plan does not say so")
    return sorted(set(problems))

<details><summary>Hint</summary>

The slide *Checkers Written in Code* and module 05 demo07b · checker-in-code check the same kind
of properties, with `in` on a string and `"out of stock" in line`. The first two blanks are one
`not in` test each ; `in` on a dictionary tests its keys. The third joins two tests with `and` :
`stock[part]` is 0, and `"out of stock"` is not in `line.lower()`.

</details>

<details><summary>Solution</summary>

The completed cell :

```python
import re


PART = re.compile(r"\b[A-Z]{2}-[A-Z]{2}-\d{2}\b")


def check_plan(plan: str, site_id: str) -> list[str]:
    with sqlite3.connect(f"file:{DB.as_posix()}?mode=ro", uri=True) as db:
        serials = [s for (s,) in db.execute("SELECT serial FROM devices WHERE site_id = ?", (site_id,))]
        stock = dict(db.execute("SELECT part_no, stock FROM parts"))
    problems = [f"device {s} is missing" for s in serials if s not in plan]
    for line in plan.splitlines():
        for part in PART.findall(line):
            if part not in stock:
                problems.append(f"part {part} does not exist")
            elif stock[part] == 0 and "out of stock" not in line.lower():
                problems.append(f"part {part} has no stock and the plan does not say so")
    return sorted(set(problems))
```

</details>

In [ ]:
GOOD = """Visit plan S04
- MR150-2024-0098 : WO-9003 needs cold head MR-CH-02, out of stock at Venlo
- UX20-2025-2311 : probe cleaning check, bring fan filter UX-FF-03"""
BAD = """Visit plan S04
- MR150-2024-0098 : bring cold head MR-CH-02 and filter MR-XX-99"""
print("good :", check_plan(GOOD, "S04"))
print("bad  :", check_plan(BAD, "S04"))
assert check_plan(GOOD, "S04") == [], "the good plan should pass"
assert len(check_plan(BAD, "S04")) == 3, "the bad plan has three problems"
print("check : pass")

## Step 3 : Parallel specialists

This step connects the pieces : devices from the database, two specialists in parallel, then a
plan and a checker in a loop. The graph is the Parallel Agents workflow of module 05. No
classifier chooses the specialists, because every visit needs both.

Most of the long cell is given. `agent_by_hand` builds each specialist from a model node and a
tool node, `specialist` wraps it as a graph node that adds one entry to `sections`, and the
parts agent receives its query in `parts_question`. `assemble` writes the plan, `check` runs
your `check_plan`, and `rounds` counts the plans written.

Four places are blank :

- `manuals_question` : ask which maintenance intervals and service bulletins apply to the
  devices in `state["devices"]`, and include that device list in the text
- `dispatch` : return a list of two `Send`s, one per specialist. Each takes a node name and a
  dictionary with the one key of `Task`, filled by the matching question function
- the blank line : one edge from both specialists into `assemble`, the edge that waits for both
  branches
- `after_check` : go back to `assemble` while problems remain and `rounds` is under 2, and end
  the graph otherwise. The `add_conditional_edges` line for `check` lists the two allowed values

Python runs the lines of a function only when the function is called, so the end of the cell
calls all three functions on `sample`, a small made-up state, with no model call. A `NameError`
in this cell means a blank is still empty, and `planner built` prints only when none is. The
check cell then tests what `dispatch` and `after_check` return, and it looks for `interval`,
`bulletin` and the serial in the manuals question.

In [ ]:
class Visit(TypedDict):
    site_id: str
    devices: str
    sections: Annotated[list[str], operator.add]
    plan: str
    problems: list[str]
    rounds: int


class Task(TypedDict):
    question: str


def agent_by_hand(model, tools, prompt):
    bound = model.bind_tools(tools)

    def model_node(state):
        return {"messages": [bound.invoke([SystemMessage(prompt), *state["messages"]])]}

    graph = StateGraph(MessagesState)
    graph.add_node("model", model_node)
    graph.add_node("tools", ToolNode(tools))
    graph.add_edge(START, "model")
    graph.add_conditional_edges("model", tools_condition)
    graph.add_edge("tools", "model")
    return graph.compile()


def ask(question):
    return {"messages": [HumanMessage(question)]}


def last_text(result):
    for message in reversed(result["messages"]):
        if isinstance(message, AIMessage) and message.content and not message.tool_calls:
            return str(message.content)
    return str(result["messages"][-1].content)


def specialist(name, worker):
    def node(state: Task) -> dict:
        try:
            answer = last_text(worker.invoke(ask(state["question"]), {"recursion_limit": 10}))
        except GraphRecursionError:
            answer = "(no answer : the specialist hit its step limit)"
        print(name, ":", answer[:300])
        return {"sections": [name + " :\n" + answer]}
    return node


PLAN_RULES = ("One bullet per device, starting with its serial number. Bring only the part that the open "
              "work order's error and summary point to. Write 'out of stock' next to any part with stock 0.")


def devices(state: Visit) -> dict:
    with sqlite3.connect(f"file:{DB.as_posix()}?mode=ro", uri=True) as db:
        rows = db.execute(
            "SELECT serial, model, firmware FROM devices WHERE site_id = ?", (state["site_id"],)).fetchall()
    return {"devices": "\n".join(f"{serial} {model} firmware {firmware}" for serial, model, firmware in rows)}


def manuals_question(state: Visit) -> str:
    return ____


def parts_question(state: Visit) -> str:
    return (
        "List the open work orders at the site and the spare parts for each device's model. "
        "A status other than closed still counts. Run this SELECT once and report every row:\n"
        "SELECT w.wo_id, w.serial, w.status, w.error_code, w.summary, p.part_no, p.name, p.stock, p.depot "
        "FROM work_orders w JOIN devices d ON d.serial = w.serial "
        "JOIN parts p ON p.model = d.model "
        f"WHERE d.site_id = '{state['site_id']}' AND w.status != 'closed'")


def dispatch(state: Visit):
    return ____


def assemble(state: Visit) -> dict:
    feedback = ""
    if state["problems"]:
        feedback = "\nFix these problems found by the checker :\n" + "\n".join(state["problems"])
    plan = metered.invoke(
        f"Write a visit plan for site {state['site_id']}. {PLAN_RULES}\n\nDevices :\n{state['devices']}\n\n"
        + "\n\n".join(state["sections"]) + feedback).content
    return {"plan": plan, "rounds": state["rounds"] + 1}


def check(state: Visit) -> dict:
    found = check_plan(state["plan"], state["site_id"])
    print(f"round {state['rounds']} :", found or "passed")
    return {"problems": found}


def after_check(state: Visit):
    return ____


parts_worker = agent_by_hand(metered, [run_sql], "Call run_sql once. Then answer from those rows and stop.")
manuals_worker = agent_by_hand(metered, [search_manuals],
                               "For each device model, call search_manuals twice : the model with 'maintenance', "
                               "then the model with 'service bulletin'. "
                               "Then answer, cite the file names, and stop.")

b = StateGraph(Visit)
b.add_node("devices", devices)
b.add_node("parts", specialist("parts", parts_worker), input_schema=Task)
b.add_node("manuals", specialist("manuals", manuals_worker), input_schema=Task)
b.add_node("assemble", assemble)
b.add_node("check", check)
b.add_edge(START, "devices")
b.add_conditional_edges("devices", dispatch, ["parts", "manuals"])
____
b.add_edge("assemble", "check")
b.add_conditional_edges("check", after_check, ["assemble", END])
planner = b.compile()

# Python runs a function body only when it is called, so call each one here. No model call.
sample = {"site_id": "S04", "devices": "MR150-2024-0098 MR-150 firmware 12.0",
          "sections": [], "plan": "", "problems": ["device missing"], "rounds": 1}
manuals_question(sample), dispatch(sample), after_check(sample)
print("planner built")

<details><summary>Hint</summary>

The slide *Router Code* and module 05 demo03b · router-parallel show how one function fans out
to several nodes with `Send`, each with its own input. The slide *Parallel Agents* and
module 05 demo06b · workflows-parallel show the edge that waits for both branches. The manuals
question asks for the intervals and bulletins and includes the device list, as `parts_question`
includes its query. `after_check` loops back while the checker reports problems and stops on a
pass or on the round cap, as module 05 demo07 · maker-checker caps its revisions.

</details>

<details><summary>Solution</summary>

The completed cell :

```python
class Visit(TypedDict):
    site_id: str
    devices: str
    sections: Annotated[list[str], operator.add]
    plan: str
    problems: list[str]
    rounds: int


class Task(TypedDict):
    question: str


def agent_by_hand(model, tools, prompt):
    bound = model.bind_tools(tools)

    def model_node(state):
        return {"messages": [bound.invoke([SystemMessage(prompt), *state["messages"]])]}

    graph = StateGraph(MessagesState)
    graph.add_node("model", model_node)
    graph.add_node("tools", ToolNode(tools))
    graph.add_edge(START, "model")
    graph.add_conditional_edges("model", tools_condition)
    graph.add_edge("tools", "model")
    return graph.compile()


def ask(question):
    return {"messages": [HumanMessage(question)]}


def last_text(result):
    for message in reversed(result["messages"]):
        if isinstance(message, AIMessage) and message.content and not message.tool_calls:
            return str(message.content)
    return str(result["messages"][-1].content)


def specialist(name, worker):
    def node(state: Task) -> dict:
        try:
            answer = last_text(worker.invoke(ask(state["question"]), {"recursion_limit": 10}))
        except GraphRecursionError:
            answer = "(no answer : the specialist hit its step limit)"
        print(name, ":", answer[:300])
        return {"sections": [name + " :\n" + answer]}
    return node


PLAN_RULES = ("One bullet per device, starting with its serial number. Bring only the part that the open "
              "work order's error and summary point to. Write 'out of stock' next to any part with stock 0.")


def devices(state: Visit) -> dict:
    with sqlite3.connect(f"file:{DB.as_posix()}?mode=ro", uri=True) as db:
        rows = db.execute(
            "SELECT serial, model, firmware FROM devices WHERE site_id = ?", (state["site_id"],)).fetchall()
    return {"devices": "\n".join(f"{serial} {model} firmware {firmware}" for serial, model, firmware in rows)}


def manuals_question(state: Visit) -> str:
    return ("For each device model, search its maintenance page and service bulletins. Which maintenance intervals and service bulletins apply to these devices?\n"
            + state["devices"])


def parts_question(state: Visit) -> str:
    return (
        "List the open work orders at the site and the spare parts for each device's model. "
        "A status other than closed still counts. Run this SELECT once and report every row:\n"
        "SELECT w.wo_id, w.serial, w.status, w.error_code, w.summary, p.part_no, p.name, p.stock, p.depot "
        "FROM work_orders w JOIN devices d ON d.serial = w.serial "
        "JOIN parts p ON p.model = d.model "
        f"WHERE d.site_id = '{state['site_id']}' AND w.status != 'closed'")


def dispatch(state: Visit):
    return [
        Send("parts", {"question": parts_question(state)}),
        Send("manuals", {"question": manuals_question(state)}),
    ]


def assemble(state: Visit) -> dict:
    feedback = ""
    if state["problems"]:
        feedback = "\nFix these problems found by the checker :\n" + "\n".join(state["problems"])
    plan = metered.invoke(
        f"Write a visit plan for site {state['site_id']}. {PLAN_RULES}\n\nDevices :\n{state['devices']}\n\n"
        + "\n\n".join(state["sections"]) + feedback).content
    return {"plan": plan, "rounds": state["rounds"] + 1}


def check(state: Visit) -> dict:
    found = check_plan(state["plan"], state["site_id"])
    print(f"round {state['rounds']} :", found or "passed")
    return {"problems": found}


def after_check(state: Visit):
    return "assemble" if state["problems"] and state["rounds"] < 2 else END


parts_worker = agent_by_hand(metered, [run_sql], "Call run_sql once. Then answer from those rows and stop.")
manuals_worker = agent_by_hand(metered, [search_manuals],
                               "For each device model, call search_manuals twice : the model with 'maintenance', "
                               "then the model with 'service bulletin'. "
                               "Then answer, cite the file names, and stop.")

b = StateGraph(Visit)
b.add_node("devices", devices)
b.add_node("parts", specialist("parts", parts_worker), input_schema=Task)
b.add_node("manuals", specialist("manuals", manuals_worker), input_schema=Task)
b.add_node("assemble", assemble)
b.add_node("check", check)
b.add_edge(START, "devices")
b.add_conditional_edges("devices", dispatch, ["parts", "manuals"])
b.add_edge(["parts", "manuals"], "assemble")
b.add_edge("assemble", "check")
b.add_conditional_edges("check", after_check, ["assemble", END])
planner = b.compile()

# Python runs a function body only when it is called, so call each one here. No model call.
sample = {"site_id": "S04", "devices": "MR150-2024-0098 MR-150 firmware 12.0",
          "sections": [], "plan": "", "problems": ["device missing"], "rounds": 1}
manuals_question(sample), dispatch(sample), after_check(sample)
print("planner built")
```

</details>

In [ ]:
sent = {item.node: item.arg["question"] for item in dispatch(sample)}
manual_q = sent.get("manuals", "").lower()
routes_ok = (set(sent) == {"parts", "manuals"} and "MR150-2024-0098" in sent.get("manuals", "")
             and "interval" in manual_q and "bulletin" in manual_q)
again_ok = after_check(sample) == "assemble"
stop_ok = after_check({**sample, "problems": []}) == END and after_check({**sample, "rounds": 2}) == END
print("nodes :", sorted(sent))
print("check :", "pass" if routes_ok and again_ok and stop_ok else "fail")

## Step 4 : Cost

The choice between several agents and one agent rests on measurements, as in module 05 demo09.
This step plans S04, then gives one agent both tools and the same task. It has no blanks, and it
is the first step that calls the model.

While the planner runs, `parts :` and `manuals :` print the start of each specialist's answer,
and the `round` lines print the checker's result. Each design then prints one `metrics :` line
with calls, tokens and seconds, and the numbers change from run to run.

Read `planner problems :`, `rounds` and `one agent problems :`, because the check cell confirms
only that a plan and metrics exist. `metered` does not retry, so one failed request stops the
cell with an error, and a `GraphRecursionError` means the single agent ran out of steps. Run the
cell again in both cases.

In [ ]:
def run_planner(site):
    return planner.invoke(
        {"site_id": site, "devices": "", "sections": [], "plan": "", "problems": [], "rounds": 0},
        {"recursion_limit": 12})


planner_result = measured("planner", lambda: run_planner("S04"))
print(planner_result.get("plan", "")[:800])
print("planner problems :", planner_result.get("problems"), "| rounds :", planner_result.get("rounds"))
print("sections :", [section.split(" :")[0] for section in planner_result.get("sections", [])])

single = create_agent(
    metered,
    tools=[run_sql, search_manuals],
    system_prompt="Write a visit plan for site S04. " + PLAN_RULES
    + " work_orders has no site_id; join it to devices on serial. A status other than closed still counts. "
    + "Use run_sql and search_manuals, then stop.")
single_result = measured(
    "one agent",
    lambda: single.invoke({"messages": [HumanMessage("Plan the visit to site S04.")]}, {"recursion_limit": 8}))
answer = last_text(single_result)
print(answer[:800])
print("one agent problems :", check_plan(answer, "S04"))

In [ ]:
ok = bool(planner_result.get("plan")) and len(metrics) >= 2 and all(row["calls"] >= 1 for row in metrics[-2:])
print("check :", "pass" if ok else "fail")
for row in metrics[-2:]:
    print(row["pattern"], row["calls"], "calls", row["input_tokens"] + row["output_tokens"], "tokens")

## If time permits

- Plan site S01 with `run_planner` and check that the part matches the CT-400's work order
- Remove "Write 'out of stock' ..." from `PLAN_RULES`, run the planner again and read round 2
- Add a fourth check : every CT-400 on firmware below 5.2 gets the 12-month fan interval
- Replace the parallel dispatch with a handoff. The transfer tool refuses a transfer that
  shares its turn with other tool calls and forwards the full worker history, as in
  demo04b · handoffs-fixed